#### DATA PREPARATION

##### 02.1 DOCUMENTAÇÃO DO DATA PREPARATION

###### Objetivo

O *Data Preparation* tem como objetivo preparar e padronizar o DataFrame de entrada para utilização pelos diferentes Profiles de qualidade de dados.

Essa etapa funciona como uma camada intermediária entre a ingestão da fonte de dados e as análises de qualidade.

O processo identifica automaticamente as características estruturais do DataFrame, classifica os tipos de colunas e disponibiliza informações técnicas que serão reutilizadas pelos Profiles seguintes.

Dessa forma, evita-se que cada Profile precise repetir operações de identificação, validação e classificação das colunas.

###### Fluxo

Fonte de dados

↓

Data Ingestion

↓

Data Preparation

↓

Schema Profile

↓

Distribution Profile

↓

Pattern Profile

↓

Demais Profiles

↓

Data Quality Score

###### Característica

O Data Preparation não depende da origem física dos dados.

Os Profiles recebem um DataFrame Spark preparado e não precisam conhecer se a origem é CSV, Parquet, Delta, banco de dados, API ou qualquer outra fonte.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- DataFrame preparado;
- informações do schema;
- classificação dos tipos de coluna;
- identificação das colunas analisáveis;
- configurações globais do framework.

In [0]:
%run "./01_DATA_INGESTION"

In [0]:
# ============================================================
# 02.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame disponibilizado pela etapa de ingestão está disponível e possui estrutura suficiente para iniciar o processo de Data Quality.

# COMO FAZ:
# Verifica a existência do objeto df, a quantidade de colunas e a existência de registros no DataFrame.

# POR QUE É IMPORTANTE:
# Evita que os Profiles seguintes sejam executados sobre um DataFrame inexistente ou estruturalmente inválido.

# PERGUNTA RESPONDIDA:
# "O DataFrame está disponível e possui estrutura válida?"

if "df" not in locals():
    raise ValueError(
        "O DataFrame 'df' não foi disponibilizado pelo Data Ingestion."
    )

if len(df.columns) == 0:
    raise ValueError(
        "O DataFrame não possui colunas."
    )

total_colunas = len(df.columns)

total_registros = df.count()

if total_registros == 0:
    raise ValueError(
        "O DataFrame não possui registros."
    )

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros}")
print(f"Total de colunas: {total_colunas}")

In [0]:
# ============================================================
# 02.4 IDENTIFICAÇÃO DO SCHEMA
# ============================================================

# O QUE FAZ:
# Identifica automaticamente o schema do DataFrame e organiza as informações estruturais de cada coluna.

# COMO FAZ:
# Percorre o schema Spark e registra nome, tipo de dado e posição de cada coluna.

# POR QUE É IMPORTANTE:
# Permite que os Profiles seguintes adaptem suas análises automaticamente aos diferentes tipos de dados existentes.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura técnica do DataFrame?"

schema_info = []

for posicao, campo in enumerate(df.schema.fields):

    schema_info.append({
        "posicao": posicao,
        "coluna": campo.name,
        "tipo_spark": campo.dataType.simpleString(),
        "tipo_detalhado": campo.dataType.typeName(),
        "nullable": campo.nullable
    })

schema_df = spark.createDataFrame(schema_info)

display(schema_df)

In [0]:
# ============================================================
# 02.5 CLASSIFICAÇÃO DOS TIPOS DE COLUNA
# ============================================================

# O QUE FAZ:
# Classifica automaticamente cada coluna em uma categoria analítica padronizada para utilização pelos Profiles.

# COMO FAZ:
# Utiliza os tipos nativos do Spark para agrupar as colunas em categorias como numérica, texto, data, data/hora e booleana.

# POR QUE É IMPORTANTE:
# Permite que cada Profile identifique automaticamente quais análises são aplicáveis a cada coluna sem depender do nome ou da origem dos dados.

# PERGUNTA RESPONDIDA:
# "Que tipo de análise pode ser aplicada a cada coluna?"

from pyspark.sql.types import (
    NumericType,
    StringType,
    DateType,
    TimestampType,
    BooleanType
)

colunas_numericas = []
colunas_texto = []
colunas_data = []
colunas_data_hora = []
colunas_booleanas = []
colunas_outros = []

for campo in df.schema.fields:

    if isinstance(campo.dataType, NumericType):

        colunas_numericas.append(campo.name)

    elif isinstance(campo.dataType, StringType):

        colunas_texto.append(campo.name)

    elif isinstance(campo.dataType, DateType):

        colunas_data.append(campo.name)

    elif isinstance(campo.dataType, TimestampType):

        colunas_data_hora.append(campo.name)

    elif isinstance(campo.dataType, BooleanType):

        colunas_booleanas.append(campo.name)

    else:

        colunas_outros.append(campo.name)


print("CLASSIFICAÇÃO DAS COLUNAS")
print("-" * 50)

print(f"Colunas numéricas: {len(colunas_numericas)}")
print(f"Colunas de texto: {len(colunas_texto)}")
print(f"Colunas de data: {len(colunas_data)}")
print(f"Colunas de data/hora: {len(colunas_data_hora)}")
print(f"Colunas booleanas: {len(colunas_booleanas)}")
print(f"Outros tipos: {len(colunas_outros)}")

In [0]:
# ============================================================
# 02.6 IDENTIFICAÇÃO DAS COLUNAS ANALISÁVEIS
# ============================================================

# O QUE FAZ:
# Define automaticamente quais Profiles podem ser aplicados a cada categoria de coluna.

# COMO FAZ:
# Utiliza a classificação dos tipos de dados criada anteriormente para construir listas de aplicabilidade analítica.

# POR QUE É IMPORTANTE:
# Evita análises inadequadas e permite que o framework seja genérico para diferentes estruturas de dados.

# PERGUNTA RESPONDIDA:
# "Quais análises podem ser executadas para cada tipo de coluna?"

colunas_distribution = (
    colunas_numericas
    + colunas_texto
    + colunas_data
    + colunas_data_hora
    + colunas_booleanas
)

colunas_pattern = (
    colunas_texto
)

colunas_outlier = (
    colunas_numericas
)

colunas_correlation = (
    colunas_numericas
)

colunas_schema = (
    df.columns
)

print("APLICABILIDADE DOS PROFILES")
print("-" * 50)

print(f"Schema Profile: {len(colunas_schema)} colunas")
print(f"Distribution Profile: {len(colunas_distribution)} colunas")
print(f"Pattern Profile: {len(colunas_pattern)} colunas")
print(f"Outlier Profile: {len(colunas_outlier)} colunas")
print(f"Correlation Profile: {len(colunas_correlation)} colunas")

In [0]:
# ============================================================
# 02.7 CONFIGURAÇÕES DO FRAMEWORK
# ============================================================

# O QUE FAZ:
# Centraliza os parâmetros utilizados pelos diferentes Profiles do framework de Data Quality.

# COMO FAZ:
# Define valores de configuração em um único local para que os Profiles possam reutilizá-los sem duplicação de parâmetros.

# POR QUE É IMPORTANTE:
# Facilita a manutenção, padroniza as análises e evita que diferentes Profiles utilizem parâmetros inconsistentes.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros controlam o comportamento do framework?"

# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================

FRAMEWORK_NAME = "Data Quality Framework"

# ============================================================
# DISTRIBUTION PROFILE
# ============================================================

TOP_N = 5

TOP_CONCENTRATION_N = [1, 3, 5]

# ============================================================
# PATTERN PROFILE
# ============================================================

PATTERN_TOP_N = 5

# ============================================================
# DATA QUALITY SCORE
# ============================================================

SCORE_MIN = 0
SCORE_MAX = 100

# ============================================================
# EXECUÇÃO
# ============================================================

DISPLAY_RESULTS = True

print("Configurações do framework carregadas.")

In [0]:
# ============================================================
# 02.8 DATAFRAME PREPARADO
# ============================================================

# O QUE FAZ:
# Define o DataFrame preparado que será utilizado como entrada pelos diferentes Profiles de qualidade.

# COMO FAZ:
# Mantém o DataFrame original da ingestão como base e cria uma referência padronizada para as etapas analíticas.

# POR QUE É IMPORTANTE:
# Estabelece um contrato único entre o Data Preparation e os Profiles seguintes, reduzindo a necessidade de repetir preparações técnicas.

# PERGUNTA RESPONDIDA:
# "Qual DataFrame será utilizado pelo framework de qualidade?"

df_prepared = df

print("DataFrame preparado com sucesso.")
print(f"Registros: {total_registros}")
print(f"Colunas: {len(df_prepared.columns)}")